# First CMO Observation Feature Extraction Demo

This notebook demonstrates the repository's CMO observation parser/Neo4j ingest shape and `feature_extraction.py` feature contract for the first `PY_CONTACT_LOG` observation in `LuaHistory_2026-06-23.txt`.

It is intentionally safe to run without a live Neo4j instance: the notebook derives the same IDs and graph payloads that `cmo_observations_txt_neo4j_ingest.ipynb` would write, then uses a small in-memory session double to exercise `extract_features_with_session` and `feature_logit`.


In [1]:
from dataclasses import asdict
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "combat_id_calibration").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from combat_id_calibration.cmo_observation_ingest import parse_observations
from combat_id_calibration.feature_extraction import (
    FeatureRequest,
    extract_features_with_session,
    feature_logit,
)
from combat_id_calibration.graph_ingest import stable_id

lua_history_path = REPO_ROOT / "LuaHistory_2026-06-23.txt"
lua_history_path


WindowsPath('C:/Users/theon/CMO-Sensor-Fusion/LuaHistory_2026-06-23.txt')

## Parse the first observation

`cmo_observations_txt_neo4j_ingest.ipynb` parses the Lua history into `EmissionObservation` objects. The first parsed object is shown below as plain JSON so it is easy to compare with the notebook output.


In [2]:
with lua_history_path.open("r", encoding="utf-8-sig", errors="replace") as handle:
    observations = list(parse_observations(handle))

first_observation = observations[0]
first_record = asdict(first_observation)
print(f"Parsed {len(observations)} observation(s)")
print(json.dumps(first_record, indent=2, sort_keys=True))


Parsed 19 observation(s)
{
  "emission_age": 33.900054931641,
  "emission_altitude": 10316.349609375,
  "emission_classificationlevel": 2,
  "emission_heading": 331.40036010742,
  "emission_latitude": 44.640933862914,
  "emission_longitude": 31.890743606263,
  "emission_role": 2122,
  "emission_sensor_name": "Slot Back [N-010 Zhuk-M]",
  "emission_solid": true,
  "emission_speed": 479.64691162109,
  "emission_target_type": "Type: Multirole (Fighter/Attack)",
  "emission_type": 2001,
  "observation_id": "13e4329885a9f92e",
  "schema": "cmo_emission_observation_v1",
  "sensor_aircraft": "Typhoon FGR.4",
  "source": "cmo_lua",
  "source_line": 1,
  "time": 1844772240
}


## Derive the graph nodes and relationships written by the ingest notebook

The ingest route writes the observation into the evidence graph with `Observation`, `Contact`, `Platform`, `Sensor`, `Emission`, `Source`, and optional `PlatformClass` nodes. These IDs mirror the deterministic IDs produced during `populate_observations_neo4j`.


In [3]:
contact_id = stable_id("contact", first_observation.observation_id)
sensor_id = stable_id("sensor", first_observation.emission_sensor_name.lower())
platform_id = stable_id("platform", first_observation.sensor_aircraft.lower())
emission_id = stable_id("emission", first_observation.observation_id, first_observation.emission_sensor_name.lower())
class_id = stable_id("platform-class", first_observation.emission_target_type.lower())
source_id = stable_id("source", first_observation.source, str(first_observation.source_line or ""))

ingest_projection = {
    "nodes": [
        {"label": "Observation", "id": first_observation.observation_id, "properties": first_record},
        {"label": "Contact", "id": contact_id, "name": f"Contact observed at {first_observation.time} by {first_observation.sensor_aircraft}"},
        {"label": "Platform", "id": platform_id, "name": first_observation.sensor_aircraft},
        {"label": "Sensor", "id": sensor_id, "name": first_observation.emission_sensor_name},
        {"label": "Emission", "id": emission_id, "sensor_name": first_observation.emission_sensor_name, "type": first_observation.emission_type, "role": first_observation.emission_role},
        {"label": "Source", "id": source_id, "source_type": first_observation.source, "locator": f"line:{first_observation.source_line}"},
        {"label": "PlatformClass", "id": class_id, "name": first_observation.emission_target_type},
    ],
    "relationships": [
        ["Contact", contact_id, "HAS_OBSERVATION", "Observation", first_observation.observation_id],
        ["Observation", first_observation.observation_id, "OBSERVED_BY", "Platform", platform_id],
        ["Platform", platform_id, "HAS_SENSOR", "Sensor", sensor_id],
        ["Contact", contact_id, "EMITTED", "Emission", emission_id],
        ["Emission", emission_id, "DETECTED_BY", "Sensor", sensor_id],
        ["Observation", first_observation.observation_id, "DERIVED_FROM", "Source", source_id],
        ["Contact", contact_id, "CLASSIFIED_AS", "PlatformClass", class_id],
    ],
}
print(json.dumps(ingest_projection, indent=2, sort_keys=True))


{
  "nodes": [
    {
      "id": "13e4329885a9f92e",
      "label": "Observation",
      "properties": {
        "emission_age": 33.900054931641,
        "emission_altitude": 10316.349609375,
        "emission_classificationlevel": 2,
        "emission_heading": 331.40036010742,
        "emission_latitude": 44.640933862914,
        "emission_longitude": 31.890743606263,
        "emission_role": 2122,
        "emission_sensor_name": "Slot Back [N-010 Zhuk-M]",
        "emission_solid": true,
        "emission_speed": 479.64691162109,
        "emission_target_type": "Type: Multirole (Fighter/Attack)",
        "emission_type": 2001,
        "observation_id": "13e4329885a9f92e",
        "schema": "cmo_emission_observation_v1",
        "sensor_aircraft": "Typhoon FGR.4",
        "source": "cmo_lua",
        "source_line": 1,
        "time": 1844772240
      }
    },
    {
      "id": "edaddfce9969c871",
      "label": "Contact",
      "name": "Contact observed at 1844772240 by Typhoon FGR.4

## Run `feature_extraction.py` against the first observation's contact

A live run would call `extract_features_neo4j` with Neo4j credentials. For a portable demonstration, this cell uses a session double that returns the same fields as `FEATURE_EXTRACTION_CYPHER`, allowing `extract_features_with_session`, feature normalization, deterministic `evidence_query_id`, and baseline `feature_logit` to run unchanged.


In [4]:
class SingleRowResult:
    def __init__(self, row):
        self._row = row
    def single(self):
        return self._row

class DemoSession:
    def __init__(self, row):
        self.row = row
        self.last_parameters = None
    def run(self, query, **parameters):
        self.last_parameters = parameters
        return SingleRowResult(self.row)

request = FeatureRequest(
    scenario_id="LuaHistory_2026-06-23",
    contact_id=contact_id,
    observation_time=str(first_observation.time),
    hypothesis=first_observation.emission_target_type,
)

demo_row = {
    "supporting_path_count": 1,
    "contradicting_path_count": 0,
    "mean_source_reliability": 0.5,
    "recency": 1.0 / (1.0 + first_observation.emission_age),
    "shortest_path_to_platform_class": 1,
    "emission_match_score": 0.0,
    "kinematic_match_score": 0.5,
    "contradiction_score": 0.0,
}

session = DemoSession(demo_row)
features = extract_features_with_session(session, request)
feature_record = features.to_record() | {"feature_logit": feature_logit(features)}

print("Cypher parameters:")
print(json.dumps(session.last_parameters, indent=2, sort_keys=True))
print("\nFeature record:")
print(json.dumps(feature_record, indent=2, sort_keys=True))


Cypher parameters:
{
  "contact_id": "edaddfce9969c871",
  "hypothesis": "Type: Multirole (Fighter/Attack)",
  "observation_time": "1844772240"
}

Feature record:
{
  "contact_id": "edaddfce9969c871",
  "contradicting_path_count": 0.0,
  "contradiction_score": 0.0,
  "emission_match_score": 0.0,
  "evidence_query_id": "ae4123173e549693",
  "feature_logit": 1.3143266250147558,
  "hypothesis": "Type: Multirole (Fighter/Attack)",
  "kinematic_match_score": 0.5,
  "mean_source_reliability": 0.5,
  "observation_time": "1844772240",
  "recency": 0.028653250029511627,
  "scenario_id": "LuaHistory_2026-06-23",
  "schema": "graph_neighbourhood_features_v1",
  "shortest_path_to_platform_class": 1.0,
  "supporting_path_count": 1.0
}


In [5]:
from combat_id_calibration.feature_extraction import extract_features_neo4j
live_features = extract_features_neo4j([request], uri="bolt://localhost:7687", user="neo4j", password="password123")


Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `CandidateIdentity` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=68, offset=171>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 171, 'line': 4, 'column': 68}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (contact:Contact {id: $contact_id})\nOPTIONAL MATCH support_path = (contact)-[*1..4]-(hypothesis)\nWHERE (hypothesis:Entity OR hypothesis:PlatformClass OR hypothesis:CandidateIdentity)\n  AND coalesce(hypothesis.name, hypothesis.id) = $hypothesis\n  AND any(rel IN relationships(support_path) WHERE type(r

In [ ]:
## Optional live Neo4j variant

After running `cmo_observations_txt_neo4j_ingest.ipynb` against a Neo4j database, replace the demo session above with:

```python
from combat_id_calibration.feature_extraction import extract_features_neo4j
live_features = extract_features_neo4j([request], uri="bolt://localhost:7687", user="neo4j", password="<password>")
```

The request should use the `contact_id` derived in this notebook for the first observation.
